# Text Generation using Vanilla RNN, LSTM, and GRU

**Problem Statement:**
Design and implement deep learning models capable of learning the underlying structure,
grammar, and contextual dependencies of a text corpus to generate coherent and meaningful
text sequences.

Three recurrent architectures are trained on the same corpus and compared:
- **Vanilla RNN** — baseline sequential model, no gating
- **LSTM** — input, forget, and output gates for long-term memory
- **GRU** — reset and update gates; simpler than LSTM, usually comparable in performance

Metrics: training loss, accuracy, and quality of generated text from seed phrases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

## 1. Text Corpus

Custom corpus on machine learning topics — small enough to train quickly but with
enough sentence variety to actually test whether the models learn contextual patterns.

In [ ]:
corpus = '''machine learning is a field of artificial intelligence
neural networks are inspired by the human brain structure
deep learning uses multiple layers to learn complex features
recurrent neural networks process sequential data across time steps
long short term memory networks solve the vanishing gradient problem
gated recurrent units are a simplified form of lstm
text generation treats language as a next word prediction task
word embeddings map words to dense vectors in a continuous space
the embedding layer converts token indices into trainable vectors
training data determines what patterns a model can learn
language models estimate the probability distribution over word sequences
deep learning has achieved strong results on natural language tasks
natural language processing enables machines to understand human language
sequence models capture temporal relationships in ordered data
attention mechanisms allow models to focus on the most relevant context
transformers build on attention to model sequences without recurrence
'''

lines = [line.strip() for line in corpus.strip().splitlines() if line.strip()]
total_words = len(corpus.split())
print(f'Corpus: {len(lines)} sentences, {total_words} total word tokens')
print()
print('First 4 sentences:')
for i, line in enumerate(lines[:4]):
    print(f'  [{i+1}] {line}')
print('  ...')

## 2. Preprocessing

**Method:** N-gram prefix sequences for next-word prediction.

Each sentence is broken into all of its prefixes, with the next token as the label:

```
"deep learning uses multiple"
  → [deep, learning]           -> uses
  → [deep, learning, uses]     -> multiple
  → [deep, learning, uses, multiple] -> layers  ...
```

All sequences are left-padded to the same length. The last token is the prediction
target `y`.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

vocab_size = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {vocab_size} unique tokens (including padding at index 0)')
print()
print('Sample word -> index mappings:')
for word, idx in list(tokenizer.word_index.items())[:8]:
    print(f'  {word:<25} -> {idx}')

In [ ]:
all_sequences = []
for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        all_sequences.append(token_list[:i + 1])

max_seq_len = max(len(s) for s in all_sequences)
print(f'Total n-gram sequences: {len(all_sequences)}')
print(f'Max sequence length   : {max_seq_len} tokens')

padded = np.array(pad_sequences(all_sequences, maxlen=max_seq_len, padding='pre'))
X = padded[:, :-1]   # input: all tokens except last
y = padded[:, -1]    # target: last token

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
# Verify the input-target pairs decode correctly
print('Sanity check — a few decoded examples:')
print()
for i in [0, 5, 20]:
    non_pad = X[i][X[i] != 0]
    input_words  = [tokenizer.index_word.get(int(t), '?') for t in non_pad]
    target_word  = tokenizer.index_word.get(int(y[i]), '?')
    print(f'  X[{i:>2}]: {input_words}')
    print(f'    y[{i:>2}]: {target_word!r}')
    print()

## 3. Model Architecture

All three models share the same template — only the recurrent layer is swapped:

```
Embedding(vocab_size, 64)
    ↓
[ SimpleRNN(128) | LSTM(128) | GRU(128) ]
    ↓
Dropout(0.2)
    ↓
Dense(vocab_size, activation='softmax')
```

Loss: `sparse_categorical_crossentropy` (targets are integer indices, not one-hot)  
Optimizer: `Adam`  

Using identical embedding size, units, and dropout keeps the comparison fair.

In [ ]:
rnn_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    SimpleRNN(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
], name='VanillaRNN')

rnn_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
rnn_model.summary()

In [ ]:
lstm_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
], name='LSTM')

lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
lstm_model.summary()

In [ ]:
gru_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    GRU(128),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
], name='GRU')

gru_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
gru_model.summary()

## 4. Training

150 epochs, batch size 32. Verbose is set to 0 to suppress per-epoch output; final
metrics are printed after each model finishes.

In [ ]:
print('Training Vanilla RNN...')
rnn_history = rnn_model.fit(X, y, epochs=150, batch_size=32, verbose=0)

loss_r = rnn_history.history['loss'][-1]
acc_r  = rnn_history.history['accuracy'][-1]
print(f'Done  ->  Loss: {loss_r:.4f}  |  Accuracy: {acc_r:.4f}')

In [ ]:
print('Training LSTM...')
lstm_history = lstm_model.fit(X, y, epochs=150, batch_size=32, verbose=0)

loss_l = lstm_history.history['loss'][-1]
acc_l  = lstm_history.history['accuracy'][-1]
print(f'Done  ->  Loss: {loss_l:.4f}  |  Accuracy: {acc_l:.4f}')

In [ ]:
print('Training GRU...')
gru_history = gru_model.fit(X, y, epochs=150, batch_size=32, verbose=0)

loss_g = gru_history.history['loss'][-1]
acc_g  = gru_history.history['accuracy'][-1]
print(f'Done  ->  Loss: {loss_g:.4f}  |  Accuracy: {acc_g:.4f}')

In [ ]:
print()
print('Final Results — Epoch 150')
print('-' * 50)
print(f'{"Model":<16}  {"Loss":>10}  {"Accuracy":>10}')
print('-' * 50)

entries = [
    ('Vanilla RNN', rnn_history),
    ('LSTM',        lstm_history),
    ('GRU',         gru_history),
]
for name, h in entries:
    l = h.history['loss'][-1]
    a = h.history['accuracy'][-1]
    print(f'{name:<16}  {l:>10.4f}  {a:>10.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Training loss
axes[0].plot(rnn_history.history['loss'],  label='Vanilla RNN', linewidth=1.5)
axes[0].plot(lstm_history.history['loss'], label='LSTM',        linewidth=1.5)
axes[0].plot(gru_history.history['loss'],  label='GRU',         linewidth=1.5)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Training accuracy
axes[1].plot(rnn_history.history['accuracy'],  label='Vanilla RNN', linewidth=1.5)
axes[1].plot(lstm_history.history['accuracy'], label='LSTM',        linewidth=1.5)
axes[1].plot(gru_history.history['accuracy'],  label='GRU',         linewidth=1.5)
axes[1].set_title('Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Text Generation

**Temperature sampling** is used instead of greedy argmax.

Argmax always picks the single highest-probability word, which produces identical
output every time and quickly becomes repetitive. Temperature scales the distribution
before sampling:

- `temp = 0.5` → sharper peaks → safer, more predictable choices
- `temp = 0.8` → mild randomness, usually readable
- `temp = 1.2` → flatter distribution → more creative, can get incoherent

This makes it easier to see what each model has actually learned about word
co-occurrence and sequence structure.

In [ ]:
def generate_text(model, seed, next_words=5, temperature=0.8):
    result = seed.lower()
    for _ in range(next_words):
        token_ids = tokenizer.texts_to_sequences([result])[0]
        token_ids = pad_sequences([token_ids], maxlen=max_seq_len - 1, padding='pre')

        # Raw probability distribution over vocabulary
        probs = model.predict(token_ids, verbose=0)[0].astype('float64')

        # Apply temperature scaling
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs)
        probs /= probs.sum()

        predicted_id   = np.random.choice(len(probs), p=probs)
        predicted_word = tokenizer.index_word.get(predicted_id, '')

        if predicted_word:
            result += ' ' + predicted_word
    return result

In [ ]:
seeds        = ['deep learning', 'recurrent neural', 'language models', 'the embedding layer']
temperatures = [0.5, 0.8, 1.2]

for seed in seeds:
    print(f"Seed: '{seed}'")
    print('-' * 65)
    for temp in temperatures:
        r = generate_text(rnn_model,  seed, next_words=5, temperature=temp)
        l = generate_text(lstm_model, seed, next_words=5, temperature=temp)
        g = generate_text(gru_model,  seed, next_words=5, temperature=temp)
        print(f'  temp={temp}')
        print(f'    RNN : {r}')
        print(f'    LSTM: {l}')
        print(f'    GRU : {g}')
    print()

## 6. Observations

**Training convergence:**

Vanilla RNN ends with the highest loss in most runs. Backpropagation through time
(BPTT) multiplies gradients by the recurrent weight matrix at every time step — if
the spectral radius of that matrix is below 1, gradients vanish exponentially. The
model ends up learning almost entirely from the last 2–3 tokens, which isn't enough
to capture sentence-level patterns.

LSTM and GRU both converge faster and land at lower loss. Their gate mechanisms
create additive update paths that resist gradient vanishing. GRU often pulls ahead
early because it has fewer parameters (no separate cell state), while LSTM's
richer representation can sometimes close that gap by the end of training.

**Generated text:**

At low temperature (0.5), all three models produce grammatically safe but repetitive
continuations. At high temperature (1.2), the difference becomes visible — RNN
output degrades more noticeably because it's drawing from a weaker learned
distribution. LSTM and GRU tend to produce completions that are contextually
consistent with the seed (e.g. "recurrent neural" → "networks process sequential..."),
reflecting that they've captured multi-token dependencies.

**Why the gap exists — brief math:**

Vanilla RNN:  `h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)`

Gradient at step t-k involves `(W_h)^k`. With k large and spectral radius < 1,
this shrinks toward zero.

LSTM cell state update: `C_t = f_t ⊙ C_{t-1} + i_t ⊙ g_t`

The forget gate `f_t` allows gradients to flow back through `C_{t-1}` without repeated
multiplication by the same weight matrix. This is what enables long-range learning.

## Architecture Comparison

| | Vanilla RNN | LSTM | GRU |
|---|---|---|---|
| **Gates** | None | Input, Forget, Output | Reset, Update |
| **Cell state** | No | Yes | No |
| **Parameters (approx)** | ~33K | ~100K | ~75K |
| **Long-term memory** | Weak | Strong | Strong |
| **Vanishing gradient** | Severe | Mitigated | Mitigated |
| **Training speed** | Fastest | Slowest | Middle |
| **Typical use** | Short sequences | Complex NLP | General NLP |

*(Counts are approximate for this configuration: embedding dim=64, hidden units=128)*

## 7. Conclusion

**Vanilla RNN:** The simplest architecture but limited by vanishing gradients.
Works for short-range pattern matching but fails when the model needs to remember
context from more than a few tokens back. Useful as a conceptual baseline, rarely
as a production model.

**LSTM:** Consistently reaches lower training loss and generates more coherent text.
The cell state acts as a dedicated long-term memory channel; the three gates control
what gets written to it, what gets erased, and what gets passed to the output. The
cost is roughly 3–4x the parameters of a plain RNN for the same hidden size.

**GRU:** Achieves comparable performance with a lighter gating structure. The update
gate controls how much past state to retain; the reset gate controls how much past
state to consider when computing the new candidate state. Fewer parameters means
faster training and easier tuning — a practical default for most sequence tasks
where LSTM's extra capacity isn't necessary.

**Takeaway:** For text generation or any task with non-trivial temporal dependencies,
gated architectures (LSTM or GRU) are the minimum baseline. In practice, transformer
models have surpassed RNNs for most NLP tasks, but understanding the RNN → LSTM → GRU
progression builds the intuition needed for sequence modeling in general.